# BKG Studies

In [1]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate


In [2]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

<Client: 'tls://192.168.202.24:8786' processes=7 threads=7, memory=20.21 GiB>

- ### Adding photon cross cleaning
- ### Commit 5f79db7
- ### 10: Running for all files in signal and 500 in bkg:

In [3]:
# Settings
sig_files = 5
bkg_files = 20

vr = "10"

sig_2mu = [
    "2Mu2E_100GeV_5p0GeV_0p4mm",
    # "2Mu2E_150GeV_5p0GeV_0p27mm",
    # "2Mu2E_200GeV_5p0GeV_0p2mm",
    "2Mu2E_500GeV_5p0GeV_0p08mm",
    # "2Mu2E_800GeV_5p0GeV_0p05mm",
    "2Mu2E_1000GeV_5p0GeV_0p04mm",
]

sig_4mu = [
    "4Mu_100GeV_5p0GeV_0p4mm",
    # "4Mu_150GeV_5p0GeV_0p27mm",
    # "4Mu_200GeV_5p0GeV_0p2mm",
    "4Mu_500GeV_5p0GeV_0p08mm",
    # "4Mu_800GeV_5p0GeV_0p05mm",
    "4Mu_1000GeV_5p0GeV_0p04mm",
]

bkg = [
    "DYJetsToMuMu_M10to50",
    "DYJetsToMuMu_M50",
    "TTJets",
    "QCD_Pt15To20",
    "QCD_Pt20To30",
    "QCD_Pt30To50",
    # "QCD_Pt50To80",
    "QCD_Pt80To120",
    # "QCD_Pt120To170",
    "QCD_Pt170To300",
    # "QCD_Pt300To470",
    "QCD_Pt470To600",
    # "QCD_Pt600To800",
    # "QCD_Pt800To1000",
    "QCD_Pt1000",       
]

#cuts to be applied (slections.yaml)
channels = ["baseNoLj", "bkg_study_isopdisp",]

ch1 = channels[0]
ch2 = channels[1]

In [4]:
# processor 2mu

runner = processor.Runner(
    # executor=processor.FuturesExecutor(),
    # executor=processor.IterativeExecutor(),
    executor=processor.DaskExecutor(client=client),
    # schema=NanoAODSchema,
    schema = llpnanoaodschema.LLPNanoAODSchema,
    #maxchunks=1,
    skipbadfiles=True,
   
)

p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"],
    verbose=True,
)

# for 2mu2e
fileset_sig_2mu = utilities.make_fileset(sig_2mu, "llpNanoAOD_v2", max_files=sig_files, location_cfg="signal_2mu2e_v10.yaml")
out_sig2 = runner.run(fileset_sig_2mu, treename="Events", processor_instance=p)
out_sig2 = out_sig2["out"]
coffea.util.save(out_sig2, "outputs/BKG_sig2mu" + vr + ".coffea")

Output()

Output()

2Mu2E_1000GeV_5p0GeV_0p04mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
2Mu2E_100GeV_5p0GeV_0p4mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
2Mu2E_500GeV_5p0GeV_0p08mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb


In [8]:
# processor 4mu

runner = processor.Runner(
    # executor=processor.FuturesExecutor(),
    # executor=processor.IterativeExecutor(),
    executor=processor.DaskExecutor(client=client),
    # schema=NanoAODSchema,
    schema = llpnanoaodschema.LLPNanoAODSchema,
    #maxchunks=1,
    skipbadfiles=True,
)

p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"],
     verbose=True,
)

fileset_sig_4mu = utilities.make_fileset(sig_4mu,  "llpNanoAOD_v2", max_files = sig_files, location_cfg = "signal_4mu_v10.yaml")
out_sig4 = runner.run(fileset_sig_4mu, treename="Events", processor_instance=p)
out_sig4 = out_sig4["out"]
coffea.util.save(out_sig4, "outputs/BKG_sig4mu" + vr + ".coffea")

Output()

Output()

4Mu_1000GeV_5p0GeV_0p04mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
4Mu_100GeV_5p0GeV_0p4mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb
4Mu_500GeV_5p0GeV_0p08mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb


In [9]:
# processor bkg

runner = processor.Runner(
    # executor=processor.FuturesExecutor(),
    # executor=processor.IterativeExecutor(),
    executor=processor.DaskExecutor(client=client),
    # schema=NanoAODSchema,
    schema = llpnanoaodschema.LLPNanoAODSchema,
    #maxchunks=1,
    skipbadfiles=True,
)

p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"],
    verbose=True,
)
fileset_bkg    = utilities.make_fileset(bkg, "skimmed_llpNanoAOD_v2", max_files = bkg_files, location_cfg = "backgrounds.yaml",)
out_bkg = runner.run(fileset_bkg, treename="Events", processor_instance=p)
out_bkg = out_bkg["out"]
coffea.util.save(out_bkg, "outputs/BKG_bkg" + vr + ".coffea")

Output()

Output()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:1493 in _work_function      │
│                                                                                                  │
│   1490 │   │   │   tic = time.time()                                                             │
│   1491 │   │   │   try:                                                                          │
│   1492 │   │   │   │   if isinstance(processor_instance, ProcessorABC):                          │
│ ❱ 1493 │   │   │   │   │   out = processor_instance.process(events)                              │
│   1494 │   │   │   │   else:                                                                     │
│   1495 │   │   │   │   │   out = processor_instance(events)                                      │
│   1496 │   │   │   except Exception as e:                                                        │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/sidm/tools/sidm_processor.py:106 in process              │
│                                                                                                  │
│   103 │   │   hists = self.build_histograms()                                                    │
│   104 │   │                                                                                      │
│   105 │   │   ### define pre-lj object, lj, post-lj obj, and event cuts per channel              │
│ ❱ 106 │   │   ch_cuts = self.build_cuts()                                                        │
│   107 │   │                                                                                      │
│   108 │   │   # loop through lj reco choices and channels, treating each lj+channel pair as a    │
│   109 │   │   for channel, cuts in ch_cuts.items():                                              │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/sidm/tools/sidm_processor.py:325 in build_cuts           │
│                                                                                                  │
│   322 │   def build_cuts(self):                                                                  │
│   323 │   │   """ Make list of pre-lj object, lj, post-lj obj, and event cuts per channel"""     │
│   324 │   │                                                                                      │
│ ❱ 325 │   │   selection_menu = utilities.load_yaml(f"{BASE_DIR}/{self.selections_cfg}")          │
│   326 │   │                                                                                      │
│   327 │   │   ch_cuts = {}                                                                       │
│   328                                                                                            │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/sidm/tools/utilities.py:173 in load_yaml                 │
│                                                                                                  │
│   170 def load_yaml(cfg):                                                                        │
│   171 │   """Load yaml files and return corresponding dict"""                                    │
│   172 │   with open(cfg, encoding="utf8") as yaml_cfg:                                           │
│ ❱ 173 │   │   return yaml.safe_load(yaml_cfg)                                                    │
│   174                                                                                            │
│   175 def make_fileset(samples, ntuple_version, max_files=-1, location_cfg="signal_v8.yaml", f   │
│   176 │   """Make fileset to pass to processor.runner"""   

Exception: Failed processing file: WorkItem(dataset='DYJetsToMuMu_M10to50', filename='root://xcache//store/group/lpcmetx/SIDM/Backgrounds/2018_v2/Skims/DYJetsToMuMu_M-10to50_H2ErratumFix/skimmed_output_1002.root', treename='Events', entrystart=0, entrystop=33, fileuuid=b'-\xd8ZP \xce\x11\xf0\xb4\xb5_\xbf\xe1\x83\xbe\xef', usermeta={'is_data': False, 'skim_factor': 0.013186077643908969, 'year': '2018'}). The error was: ScannerError('while scanning a quoted scalar', <yaml.error.Mark object at 0x7f3d35f67980>, 'found unexpected end of stream', <yaml.error.Mark object at 0x7f3d35f66b10>).

In [5]:
# opening files
output_2mu = coffea.util.load("outputs/BKG_sig2mu" + vr + ".coffea")
output_4mu = coffea.util.load("outputs/BKG_sig4mu" + vr + ".coffea")
output_bkg = coffea.util.load("outputs/BKG_bkg"  +  vr  + ".coffea")

In [7]:
#printing cutflows
output_2mu[sig_2mu[0]]["cutflow"][ch1].print_table(unweighted=True)
output_4mu[sig_4mu[0]]["cutflow"][ch1].print_table(unweighted=True)
output_bkg[bkg[0]]["cutflow"][ch1].print_table(unweighted=True)

cut name         individual cut N    all cut N
-------------  ------------------  -----------
No selection              10587.0      10587.0
pass triggers               851.0        851.0
PV filter                 10587.0        851.0


In [ ]:
# plt.subplots(1, npl, figsize=(npl*figw, figh))
npl = 3

HistToPlot = ["lj_lj_invmass"]

for thisHist in HistToPlot:
    plt.subplots(1, npl, figsize=(npl*10, 6))
    plt.subplot(1, npl, 1)
    utilities.plot(output_2mu[sig_2mu[0]]["hists"][thisHist][ch1, ::2j], label = sig_2mu[0], density=True,)
    utilities.plot(output_4mu[sig_4mu[0]]["hists"][thisHist][ch1, ::2j], label = sig_4mu[0], density=True,)
    utilities.plot(output_bkg[bkg[0]]["hists"][thisHist][ch1, ::2j], label = bkg[0], density=True,)
    plt.legend(alignment="left", loc="upper right")
    
    plt.subplot(1, npl, 2)
    utilities.plot(output_2mu[sig_2mu[0]]["hists"][thisHist][ch2, ::2j], label = sig_2mu[0], density=True,)
    utilities.plot(output_4mu[sig_4mu[0]]["hists"][thisHist][ch2, ::2j], label = sig_4mu[0], density=True,)
    utilities.plot(output_bkg[bkg[0]]["hists"][thisHist][ch2, ::2j], label = bkg[0], density=True,)
    plt.legend(alignment="left", loc="upper right")

# for sss in sig_2mu:
#     utilities.plot(output_2mu[sss]["hists"]["lj_pt"][ch1, ::2j], label = sig_2mu, density=True)
#     utilities.plot(output_4mu[sss]["hists"]["lj_pt"][ch1, ::2j], label = sig_4mu, density=True)
#     plt.legend(sig_2mu, title="Sample", alignment="left", loc="upper right")
#     plt.ylabel("Arbitrary units")

# plt.subplot(1, npl, 2)
# for sss in sig_4mu:
#     utilities.plot(output_4mu[sss]["hists"]["lj_pt"][ch1, ::2j], label = sig_4mu, density=True)
#     plt.legend(sig_4mu, title="Sample", alignment="left", loc="upper right")
#     plt.ylabel("Arbitrary units")


# plt.savefig(f"plots/master_{vr}_dxy", bbox_inches="tight")


In [ ]:
npl = 2
ch = [ch1, ch2]
HistList = ["lj_n", "lj_pt", "lj0_pt", "lj1_pt", ]#, "lj_lj_invmass", "lj_lj_absdR", "lj_lj_absdphi", "lj_lj_absdeta"]

for thisHist in HistList:
    plt.subplots(1, npl, figsize=(npl*figw, figh))
    for i in range(npl):
        plt.subplot(1, npl, i+1)
    
        QCD, DYJ, TTJ = 0, 0, 0
        for sample in allsamples:
            if "QCD" in sample:
                QCD = QCD + out[sample]["hists"][thisHist][ch[i], ::2j]
                
            elif "DYJ" in sample:
                DYJ = DYJ + out[sample]["hists"][thisHist][ch[i], ::2j]
                
            elif "TTJ" in sample:
                TTJ = TTJ + out[sample]["hists"][thisHist][ch[i], ::2j]
            
            elif "2Mu2E" in sample:
                utilities.plot(out[sample]["hists"][thisHist][ch[i], ::2j], label = sig_2mu, density=True)
            
            elif "4Mu" in sample:
                utilities.plot(out[sample]["hists"][thisHist][ch[i], ::2j], label = sig_2mu, density=True)
                
        utilities.plot(QCD, label = "QCD", density=True)
        utilities.plot(DYJ, label = "DY", density=True)
        utilities.plot(TTJ, label = "TTJ", density=True)
        
        plt.legend(title="Sample", alignment="left", loc=0)
        plt.savefig(f"plots/BKG_{vr}_{thisHist}", bbox_inches="tight")

In [ ]:
# adding and saving
out_all = out_sig2 | out_sig4 | out_bkg 
coffea.util.save(out_all, "outputs/bkg_" + vr + ".coffea")

In [ ]:
# test
print(out_all.keys())

# Start plotting

In [ ]:
# opening file
output = coffea.util.load("outputs/bkg_" + vr + ".coffea")
out = output

In [ ]:
allsamples = sig_2mu + sig_4mu + bkg
print(allsamples)

figw, figh = 10, 10
# cols = ["b", "r", "g", "c", "k", "y"]
# labels = ["0.15", "0.17", "0.19", "0.21", "0.23", "0.25"]

In [ ]:
#Combined background samples

# DY_bkg = "DYJetsToMuMu_M10to50",
#     "DYJetsToMuMu_M50",
# for i, hist in allsamples:
    

In [ ]:
npl = 2
ch = [ch1, ch2]
HistList = ["lj_n", "lj_pt", "lj0_pt", "lj1_pt", ]#, "lj_lj_invmass", "lj_lj_absdR", "lj_lj_absdphi", "lj_lj_absdeta"]

for thisHist in HistList:
    plt.subplots(1, npl, figsize=(npl*figw, figh))
    for i in range(npl):
        plt.subplot(1, npl, i+1)
    
        QCD, DYJ, TTJ = 0, 0, 0
        for sample in allsamples:
            if "QCD" in sample:
                QCD = QCD + out[sample]["hists"][thisHist][ch[i], ::2j]
                
            elif "DYJ" in sample:
                DYJ = DYJ + out[sample]["hists"][thisHist][ch[i], ::2j]
                
            elif "TTJ" in sample:
                TTJ = TTJ + out[sample]["hists"][thisHist][ch[i], ::2j]
            
            elif "2Mu2E" in sample:
                utilities.plot(out[sample]["hists"][thisHist][ch[i], ::2j], label = sig_2mu, density=True)
            
            elif "4Mu" in sample:
                utilities.plot(out[sample]["hists"][thisHist][ch[i], ::2j], label = sig_2mu, density=True)
                
        utilities.plot(QCD, label = "QCD", density=True)
        utilities.plot(DYJ, label = "DY", density=True)
        utilities.plot(TTJ, label = "TTJ", density=True)
        
        plt.legend(title="Sample", alignment="left", loc=0)
        plt.savefig(f"plots/BKG_{vr}_{thisHist}", bbox_inches="tight")

In [ ]:
npl = 2
ch = [ch1, ch2]
HistList = ["lj_lj_invmass", "lj_lj_absdR", "lj_lj_absdphi", "lj_lj_absdeta"]

for thisHist in HistList:
    plt.subplots(1, npl, figsize=(npl*figw, figh))
    for i in range(npl):
        plt.subplot(1, npl, i+1)
    
        QCD, DYJ, TTJ = 0, 0, 0
        for sample in allsamples:
            if "QCD" in sample:
                QCD = QCD + out[sample]["hists"][thisHist][ch[i], ::2j]
                
            elif "DYJ" in sample:
                DYJ = DYJ + out[sample]["hists"][thisHist][ch[i], ::2j]
                
            elif "TTJ" in sample:
                TTJ = TTJ + out[sample]["hists"][thisHist][ch[i], ::2j]
            
            elif "2Mu2E" in sample:
                utilities.plot(out[sample]["hists"][thisHist][ch[i], ::2j], label = sig_2mu, density=True)
            
            elif "4Mu" in sample:
                utilities.plot(out[sample]["hists"][thisHist][ch[i], ::2j], label = sig_2mu, density=True)
                
        utilities.plot(QCD, label = "QCD", density=True)
        utilities.plot(DYJ, label = "DY", density=True)
        utilities.plot(TTJ, label = "TTJ", density=True)
        
        plt.legend(title="Sample", alignment="left", loc=0)
        plt.savefig(f"plots/BKG_{vr}_{thisHist}", bbox_inches="tight")

In [ ]:
raise SystemExit("Notebook stopped intentionally after this cell.")

# stop here:

In [ ]:
# LJ pt
npl = 3
plt.subplots(1, npl, figsize=(npl*figw, figh))
print(allsamples)
plt.subplot(1, npl, 1)
for sss in allsamples[1:]:
    # print(sss)
    utilities.plot(out[sss]["hists"]["lj_pt"][ch1, ::2j], label = sig_2mu, density=True)
    plt.legend(allsamples, title="Sample", alignment="left", loc=0)
    # plt.title("den")
    plt.ylabel("Arbitrary units")

plt.subplot(1, npl, 2)
for sss in allsamples[1:]:
    utilities.plot(out[sss]["hists"]["lj_pt"][ch2, ::2j], label = sig_2mu, density=True)
    plt.legend(allsamples, title="Sample", alignment="left", loc=0)
    # plt.title("den")
    plt.ylabel("Arbitrary units")

# plt.subplot(1, npl, 3)
# for sss in allsamples[1:]:
#     utilities.plot(out[sss]["hists"]["lj_pt"][ch3, ::2j], label = sig_2mu, density=True)
#     plt.legend(allsamples, title="Sample", alignment="left", loc=0)
#     # plt.title("den")
#     plt.ylabel("Arbitrary units")

plt.savefig(f"plots/BKG_{vr}_lj_pt", bbox_inches="tight")

In [ ]:
# LJ quantities

# "lj_lj_invmass", "lj_lj_absdphi", "lj_lj_absdR""lj_lj_absdeta"
# , "lj_lj_absdR", "lj_lj_invmass"]
hists_to_plot = ["lj_n","lj0_pt", "lj1_pt"]

for hst in hists_to_plot:
    npl = 3
    plt.subplots(1, npl, figsize=(npl*figw, figh))
    
    plt.subplot(1, npl, 1)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch1, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")
    
    plt.subplot(1, npl, 2)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch2, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")
    
    plt.subplot(1, npl, 3)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch3, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")

In [ ]:
# LJ-LJ quantities

hists_to_plot = ["lj_lj_invmass", "lj_lj_absdR", "lj_lj_absdphi", "lj_lj_absdeta"]

for hst in hists_to_plot:
    npl = 3
    plt.subplots(1, npl, figsize=(npl*figw, figh))
    
    plt.subplot(1, npl, 1)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch1, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")
    
    plt.subplot(1, npl, 2)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch2, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")
    
    plt.subplot(1, npl, 3)
    for sss in allsamples:
        utilities.plot(out[sss]["hists"][hst][ch3, ::2j], density=True)
        plt.legend(allsamples, title="Sample", alignment="left", loc=0)
        # plt.title("den")